# 03 — Base oficial de logradouros com CEP

**Objetivo:** criar a **base oficial** de logradouros com CEP em `02_dados_editados\logradouros_cep.gpkg`. É a camada do Geosanja **com todos os campos originais** + colunas de CEP e **observações dos problemas** encontrados no pareamento (etapa 02). Daqui em diante **é nesta base que você corrige os CEPs**; as etapas 05 a 07 leem o CEP direto dela.

**Regra do padrão (editado → bruto):**
- a base **não existe** → é criada aqui (versão inicial);
- a base **já existe** (com suas edições) → **não é alterada**. A versão automática é gravada só para comparação em `03_processamento\03_base_logradouros_cep\logradouros_cep_automatico.gpkg`.

**Colunas acrescentadas:**

| Coluna | Significado |
|---|---|
| `status_cep` | `pareado`, `multi_cep`, `ambiguo`, `sem_cep` ou `sem_nome` (resultado da etapa 02; filtre por aqui) |
| **`cep`** | CEP oficial do segmento. Vários CEPs na mesma coluna, separados por `\|`. **É esta coluna que você edita** |
| `cep_origem` | `automatico:<método>` ou `manual` (escreva `manual` ao editar) |
| `cep_conferido` | `Não` / `Sim` (marque `Sim` o que já verificou) |
| `obs_cep` | observação do problema, com sugestão de CEP órfão de nome parecido quando houver |
| `ceps_candidatos` | candidatos quando o nome é ambíguo (`cep` fica vazio até você decidir) |

**Entradas:** `01_dados_brutos\geosanja\logradouros.gpkg` + saídas da etapa 02.
**Saída:** `02_dados_editados\logradouros_cep.gpkg` (camada `logradouros_cep`), só na primeira vez.

**Estrutura:** 1 Preparação · 2 Funções · 3 Carregar entradas · 4 Colunas de CEP · 5 Sugestões · 6 Observações · 7 Exportação · 8 Resumo

## 1. Preparação e configuração

In [ ]:
# --- Preparação padrão: acha a raiz do projeto, lê o config.yaml e liga o log (tela + arquivo) -------------
import sys                                            # para o Python achar as funções comuns (scripts/src)
from pathlib import Path                              # caminhos de arquivos
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())   # sobe pastas até achar o config.yaml
sys.path.insert(0, str(RAIZ / "scripts"))             # libera o "from src import ..."
from src import config, registro                      # funções comuns do padrão de projetos
cfg = config.carregar(RAIZ)                           # lê o config.yaml (caminhos e parâmetros)
ETAPA = "03_base_logradouros_cep"           # nome desta etapa (= nome do notebook)
log = registro.iniciar_log(ETAPA, cfg)                # log na tela e em 03_processamento/logs/
PASTA_SAIDA = cfg.pasta_etapa(ETAPA)                  # 03_processamento/<ETAPA> (criada se não existir)
import numpy as np                    # contas com colunas
import pandas as pd                   # tabelas
import pyogrio                        # leitura/escrita de arquivos geográficos
from rapidfuzz import fuzz, process   # comparação de textos "parecidos"

# --- Entradas (config.yaml) ----------------------------------------------------------------------------
ARQ_GEOSANJA = cfg.caminho("brutos.geosanja")                       # camada original, com todos os campos
CAMADA_ORIGINAL = cfg.camada("brutos.geosanja")
ARQ_ETAPA1_GPKG = cfg.caminho("processamento.pareamento")           # resultado automático da etapa 02
CAMADA_ETAPA1 = cfg.camada("processamento.pareamento")
ARQ_PONTE = cfg.caminho("processamento.ponte")
ARQ_ORFAOS = cfg.caminho("processamento.orfaos")

# --- Saída: a base oficial (02_dados_editados) ---------------------------------------------------------------
ARQ_BASE = cfg.caminho("editados.logradouros_cep")
CAMADA_BASE = cfg.camada("editados.logradouros_cep")

# --- Parâmetros --------------------------------------------------------------------------------------------------
LIMIAR_SUGESTAO = cfg.parametro("base_logradouros.limiar_sugestao")   # similaridade mínima do nome para sugerir CEP órfão

log.info("Configuração carregada!")
for _nome, _arq in [("geosanja original", ARQ_GEOSANJA), ("Pareamento (etapa 02)", ARQ_ETAPA1_GPKG),
                    ("Ponte", ARQ_PONTE), ("Órfãos", ARQ_ORFAOS), ("Base editada", ARQ_BASE)]:
    log.info("%s: %s | existe: %s", _nome, _arq.name, _arq.exists())

## 2. Funções auxiliares
Vêm de `scripts/src` (comuns a todos os projetos).

In [ ]:
from src.validacao import validar_tabela                         # bloco padrão de validação
from src.texto import normalizar, normalizar_nome, tirar_tipo    # normalização de nomes de logradouro
log.info("Funções auxiliares carregadas!")

## 3. Carregar entradas
Lê a camada original do Geosanja (todos os campos + geometria) e junta o resultado da Etapa 02 pelo `id`.

In [ ]:
log.info("Carregando a camada original do geosanja...")
orig = pyogrio.read_dataframe(ARQ_GEOSANJA, layer=CAMADA_ORIGINAL)
n_orig = len(orig)
for col in ["id", "tipo", "nome", "loteamento"]:
    assert col in orig.columns, f"Campo '{col}' não existe na camada original"
colunas_originais = [c for c in orig.columns if c != "geometry"]
log.info("Feições: %d | CRS: %s | campos originais: %s", n_orig, orig.crs, colunas_originais)
validar_tabela("geosanja original", orig, chave=["id"], obrigatorias=["id"])

log.info("Carregando o resultado da Etapa 02...")
e1 = pyogrio.read_dataframe(ARQ_ETAPA1_GPKG, layer=CAMADA_ETAPA1, read_geometry=False,
                            columns=["id", "status", "cep", "ceps_candidatos", "metodo", "score_nome"])
validar_tabela("Etapa 02 (status por segmento)", e1, chave=["id"], obrigatorias=["id", "status"])
ponte = pd.read_parquet(ARQ_PONTE, columns=["id_geosanja", "cep", "logradouro_correios", "bairro_correios"])
orfaos = pd.read_csv(ARQ_ORFAOS, sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)
log.info("Ponte: %d pares | órfãos dos correios: %d", len(ponte), len(orfaos))

base = orig.merge(e1.rename(columns={"status": "status_cep"}), on="id", how="left", validate="one_to_one")
sem_e1 = int(base["status_cep"].isna().sum())
(log.info if sem_e1 == 0 else log.error)("Segmentos sem resultado da Etapa 02: %d", sem_e1)
assert len(base) == n_orig, "A junção mudou o número de feições!"
log.info("Status (Etapa 02):\n%s", base["status_cep"].value_counts(dropna=False).to_string())
log.info("Entradas carregadas!")

## 4. Montar as colunas de CEP
- `cep`: segmento `pareado` → o CEP; `multi_cep` → todos os CEPs separados por `|`; `ambiguo`, `sem_cep` e `sem_nome` → vazio;
- `ceps_candidatos`: só nos `ambiguo`;
- `cep_origem`: `automatico:<método>` onde há CEP;
- `cep_conferido`: `Não` para todos (você marca `Sim` ao verificar).

In [ ]:
log.info("Montando as colunas de CEP...")
st = base["status_cep"]
base["cep_final"] = np.select([st == "pareado", st == "multi_cep"], [base["cep"], base["ceps_candidatos"]], default=None)
base["candidatos_final"] = np.where(st == "ambiguo", base["ceps_candidatos"], None)
base["cep_origem"] = np.where(base["cep_final"].notna(), "automatico:" + base["metodo"].astype(str), None)
base["cep_conferido"] = "Não"

# --- Logradouro dos correios usado em cada segmento (para as observações) ---------------------------------------
corr_seg = (ponte.groupby("id_geosanja")["logradouro_correios"]
            .agg(lambda s: " / ".join(dict.fromkeys(s.dropna()))).rename("log_correios"))
base = base.merge(corr_seg, left_on="id", right_index=True, how="left")
base["log_correios"] = base["log_correios"].fillna("")
log.info("CEP preenchido: %d segmentos | multi_cep: %d | candidatos (ambíguos): %d",
         int(base["cep_final"].notna().sum()), int((st == "multi_cep").sum()), int(base["candidatos_final"].notna().sum()))
log.info("Colunas de CEP montadas!")

## 5. Sugestões de CEP para vias sem CEP
Para cada via **nomeada sem CEP**, procura entre os **CEPs órfãos dos Correios** o de **nome mais parecido** (sem o tipo). A sugestão só aparece na observação (`obs_cep`); **o CEP não é preenchido sozinho**. No empate de nome, vale o bairro mais parecido com o loteamento.

In [ ]:
log.info("Procurando CEPs órfãos de nome parecido...")
sem = base[base["status_cep"] == "sem_cep"].copy()
sem["chave_via"] = sem["tipo"].fillna("") + "|" + sem["nome"].fillna("") + "|" + sem["loteamento"].fillna("")
vias_sem = sem.drop_duplicates("chave_via")[["chave_via", "nome", "loteamento"]].copy()
vias_sem["nome_key"] = normalizar(vias_sem["nome"])
vias_sem["lote_key"] = normalizar(vias_sem["loteamento"])
vias_sem = vias_sem[vias_sem["nome_key"] != ""]

orf = orfaos.copy()
orf["nome_key"] = normalizar(orf["logradouro_orig"]).map(tirar_tipo)
orf["bairro_key"] = normalizar(orf["bairro_orig"])
orf = orf[orf["nome_key"] != ""].reset_index(drop=True)
log.info("Vias sem CEP: %d | CEPs órfãos dos correios: %d", len(vias_sem), len(orf))

q_nomes = vias_sem["nome_key"].drop_duplicates().tolist()
c_nomes = orf["nome_key"].drop_duplicates().tolist()
sugestao_por_via = {}
if q_nomes and c_nomes:
    matriz = process.cdist(q_nomes, c_nomes, scorer=fuzz.token_sort_ratio, score_cutoff=LIMIAR_SUGESTAO, workers=-1)
    pos_orf = orf.groupby("nome_key").indices
    achados = []
    for i, q in enumerate(q_nomes):                          # laço curto (só nomes distintos)
        for j in np.flatnonzero(matriz[i] > 0):
            for pos in pos_orf[c_nomes[j]]:
                achados.append((q, pos, float(matriz[i, j])))
    par = pd.DataFrame(achados, columns=["nome_key", "pos", "score_nome"])
    if len(par):
        par = par.merge(vias_sem[["chave_via", "nome_key", "lote_key"]], on="nome_key").join(
            orf[["cep", "logradouro_orig", "bairro_orig", "bairro_key"]], on="pos")
        pb = par[["lote_key", "bairro_key"]].drop_duplicates()
        pb["score_bairro"] = [fuzz.ratio(a, b) for a, b in zip(pb["lote_key"], pb["bairro_key"])]
        par = par.merge(pb, on=["lote_key", "bairro_key"], how="left")
        melhor = par.sort_values(["chave_via", "score_nome", "score_bairro"], ascending=[True, False, False]).drop_duplicates("chave_via")
        sugestao_por_via = dict(zip(melhor["chave_via"],
            "Sugestão: " + melhor["cep"] + " (" + melhor["logradouro_orig"] + ", " + melhor["bairro_orig"] + "); nome "
            + melhor["score_nome"].round(0).astype(int).astype(str) + ", bairro " + melhor["score_bairro"].round(0).astype(int).astype(str)))
base["sugestao"] = (base["tipo"].fillna("") + "|" + base["nome"].fillna("") + "|" + base["loteamento"].fillna("")).map(sugestao_por_via)
log.info("Vias sem CEP com sugestão: %d de %d", len(sugestao_por_via), len(vias_sem))
log.info("Sugestões prontas!")

## 6. Observações (`obs_cep`)
Texto curto por segmento explicando o problema ou a origem do CEP. Segmentos pareados por nome exato com bairro conferido ficam **sem observação** (são os mais confiáveis).

In [ ]:
log.info("Escrevendo as observações...")
st = base["status_cep"]
met = base["metodo"].fillna("")
score = base["score_nome"].round(0).astype("Int64").astype(str)
corr = base["log_correios"]
n_cand = base["candidatos_final"].fillna("").str.count(r"\|") + 1
n_ceps = base["cep_final"].fillna("").str.count(r"\|") + 1

obs = np.select(
    [st == "sem_nome",
     st == "sem_cep",
     st == "ambiguo",
     st == "multi_cep",
     (st == "pareado") & (met == "fuzzy"),
     (st == "pareado") & (met == "iniciais"),
     (st == "pareado") & (met == "exato_tipo_nome_unico"),
     (st == "pareado") & (met == "exato_nome_bairro")],
    ["Via sem nome no geosanja",
     np.where(base["sugestao"].notna(), "Nenhum CEP localizado para o nome | " + base["sugestao"].astype(str), "Nenhum CEP localizado para o nome"),
     "Nome com vários CEPs nos correios e o loteamento não decide (" + n_cand.astype(str) + " candidatos)",
     "Via com " + n_ceps.astype(str) + " CEPs nos correios (mantidos na coluna cep)",
     "Pareado por nome parecido (score " + score + ") com '" + corr + "'",
     "Pareado por iniciais com '" + corr + "'",
     "Nome com CEP único nos correios; loteamento não conferiu",
     "Pareado sem considerar o tipo da via (correios: '" + corr + "')"],
    default="")
base["obs_cep"] = obs
log.info("Observações por tipo (início do texto):\n%s", pd.Series(obs[obs != ""]).str.slice(0, 28).value_counts().to_string())
log.info("Segmentos com observação: %d de %d", int((obs != "").sum()), len(base))
log.info("Observações prontas!")

## 7. Exportação
Grava a base oficial com os campos originais + colunas de CEP. **Se o arquivo já existir, não sobrescreve**: cria outro com data e hora no nome (e avisa).

In [ ]:
log.info("Gravando a base oficial inicial...")
saida = base.drop(columns=["cep", "ceps_candidatos", "metodo", "score_nome", "log_correios", "sugestao"], errors="ignore")
saida = saida.rename(columns={"cep_final": "cep", "candidatos_final": "ceps_candidatos"})
colunas_saida = colunas_originais + ["status_cep", "cep", "cep_origem", "cep_conferido", "obs_cep", "ceps_candidatos", "geometry"]
saida = saida[colunas_saida]
for col in ["status_cep", "cep", "cep_origem", "cep_conferido", "obs_cep", "ceps_candidatos"]:
    saida[col] = saida[col].astype(object).where(saida[col].notna() & (saida[col] != ""), None)     # vazio vira nulo

if ARQ_BASE.exists():                                            # a base editada já existe: NUNCA é sobrescrita
    destino = PASTA_SAIDA / "logradouros_cep_automatico.gpkg"
    log.warning("A base editada %s já existe e NÃO foi alterada.", ARQ_BASE.name)
    log.warning("Versão automática gravada só para comparação em: %s", destino)
    if destino.exists():
        destino.unlink()                                         # a versão automática é regerável
else:
    destino = ARQ_BASE                                           # primeira vez: cria a base oficial em 02_dados_editados
    destino.parent.mkdir(parents=True, exist_ok=True)
    log.info("A base editada ainda não existe: criando a versão inicial em %s", destino)
assert len(saida) == n_orig, "Perda de feições na exportação!"
saida.to_file(destino, layer=CAMADA_BASE, driver="GPKG", engine="pyogrio")
log.info("Base salva: %s (%d feições, CRS %s)", destino.name, len(saida), saida.crs)
validar_tabela("Base oficial inicial", saida, chave=["id"], obrigatorias=["id", "status_cep"], linhas_entrada=n_orig)
log.info("Exportação concluída!")

## 8. Resumo

In [ ]:
log.info("Resumo final...")
nomeados = int((saida["status_cep"] != "sem_nome").sum())
com_cep = int(saida["cep"].notna().sum())
log.info("Segmentos: %d | nomeados: %d | com CEP: %d (%.1f%% dos nomeados)", len(saida), nomeados, com_cep, com_cep / nomeados * 100)
log.info("Por status:\n%s", saida["status_cep"].value_counts().to_string())
log.info("Com observação: %d | com sugestão de CEP: %d", int(saida["obs_cep"].notna().sum()), int(saida["obs_cep"].fillna("").str.contains("Sugestão").sum()))
log.info("Como usar: abra a camada no QGIS, filtre por status_cep / obs_cep, corrija 'cep' (vários = separados por |), ")
log.info("  escreva 'manual' em cep_origem e 'Sim' em cep_conferido, salve. Depois rode as etapas 05 em diante.")
log.info("Base inicial concluída!")